# 🎙️ Piper Vietnamese Voice Training Studio (Google Colab T4 GPU)

Công cụ **Huấn luyện Mô hình Giọng nói Tiếng Việt VITS** siêu tốc cho **Piper TTS Studio**.

### ⚡ Quy Trình Huấn Luyện 1-Click (Siêu Nhanh, Không Cần Biên Dịch):
1. **Chuẩn bị Dữ liệu:** Đã tạo gói `<voice_name>_dataset.zip` hoàn toàn offline trong ứng dụng TTS Studio.
2. **Cài đặt GPU:** Trên thanh menu Colab chọn **Runtime** ➔ **Change runtime type** ➔ Chọn **T4 GPU** ➔ Nhấn **Save**.
3. **Chạy các ô bên dưới:**
   - Bấm nút **Play (▶️)** ở **Bước 1** (~30 giây cài đặt môi trường).
   - Bấm nút **Play (▶️)** ở **Bước 2** để mở giao diện Web Gradio.
   - Kéo thả file `dataset.zip`, bấm **Bắt Đầu Huấn Luyện (Train)** rồi xuất file `.onnx` mang về TTS Studio!

In [ ]:
# @title 🚀 Bước 1: Khởi Tạo Môi Trường Piper Training & Tải Checkpoint Gốc Tiếng Việt (~30 giây) { display-mode: "form" }
import os, sys, subprocess, time, urllib.request
from pathlib import Path

print('======================================================================')
print('🎙️ KHỞI TẠO PIPER TRAINING ENGINE CHO GOOGLE COLAB GPU T4...')
print('======================================================================')

def download_with_progress(url, dest_path, desc='Đang tải file'):
    dest = Path(dest_path)
    if dest.exists() and dest.stat().st_size > 1000000:
        print(f'✅ {desc}: Đã có sẵn ({dest.stat().st_size / (1024*1024):.1f} MB)')
        return str(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_suffix('.download_tmp')
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    try:
        with urllib.request.urlopen(req) as resp, open(tmp, 'wb') as f:
            total_size = int(resp.headers.get('Content-Length', 0))
            downloaded = 0
            start_time = time.time()
            last_print = 0
            while True:
                chunk = resp.read(2 * 1024 * 1024)
                if not chunk: break
                f.write(chunk)
                downloaded += len(chunk)
                now = time.time()
                if now - last_print > 0.4 or downloaded == total_size:
                    last_print = now
                    elapsed = max(0.1, now - start_time)
                    speed = (downloaded / (1024 * 1024)) / elapsed
                    pct = int((downloaded / total_size) * 100) if total_size > 0 else 0
                    mb_down = downloaded / (1024 * 1024)
                    mb_tot = total_size / (1024 * 1024)
                    sys.stdout.write(f'\r-> {desc}: [{pct}%] ({mb_down:.1f} MB / {mb_tot:.1f} MB) - {speed:.1f} MB/s')
                    sys.stdout.flush()
        print('')
        tmp.rename(dest)
        print(f'✅ {desc}: Tải hoàn tất!')
        return str(dest)
    except Exception as e:
        if tmp.exists(): tmp.unlink()
        print(f'\n⚠️ Lỗi khi tải {desc}: {e}')
        return None

# 1. Gói hệ thống eSpeak-NG
print('\n-> [1/3] Cài đặt eSpeak-NG & thư viện phụ trợ...')
subprocess.run('apt-get update -qq && apt-get install -y -qq espeak-ng libespeak-ng-dev ffmpeg', shell=True, check=False)

# 2. Thư viện Python cho Piper Training & UI
print('\n-> [2/3] Cài đặt PyTorch Lightning, Piper-Phonemize, Gradio...')
subprocess.run('pip install -q gradio onnx soundfile pandas piper-phonemize pytorch-lightning==1.9.5 cython librosa onnxruntime', shell=True, check=False)

# 3. Cài đặt piper_train
print('\n-> [3/3] Cài đặt Piper VITS Training Engine (piper_train)...')
if not os.path.exists('piper'):
    subprocess.run('git clone --depth 1 https://github.com/rhasspy/piper.git', shell=True, check=False)

piper_src = os.path.abspath('piper/src/python')
if os.path.exists(piper_src):
    subprocess.run(f'cd {piper_src} && pip install -q -e .', shell=True, check=False)
    subprocess.run(f'cd {piper_src} && python3 setup.py build_ext --inplace', shell=True, check=False)
    if piper_src not in sys.path:
        sys.path.insert(0, piper_src)
    os.environ['PYTHONPATH'] = f"{piper_src}:{os.environ.get('PYTHONPATH', '')}"

# 4. Tải Checkpoint Tiếng Việt gốc để fine-tune siêu tốc
ckpt_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/epoch=4769-step=919580.ckpt'
cfg_url = 'https://huggingface.co/datasets/rhasspy/piper-checkpoints/resolve/main/vi/vi_VN/vais1000/medium/config.json'
download_with_progress(ckpt_url, 'base_checkpoint/vi_base.ckpt', desc='Mô hình Tiếng Việt Gốc (Base Checkpoint)')
download_with_progress(cfg_url, 'base_checkpoint/config.json', desc='Cấu hình Ngữ âm Tiếng Việt (Config)')

verify_cmd = f"PYTHONPATH={piper_src} python3 -c 'import piper_train; print(\"PIPER_TRAIN_READY\")'"
ver_res = subprocess.run(verify_cmd, shell=True, capture_output=True, text=True)
if 'PIPER_TRAIN_READY' in ver_res.stdout:
    print('\n======================================================================')
    print('🎉 MÔI TRƯỜNG PIPER ĐÃ SẴN SÀNG! HÃY BẤM NÚT PLAY Ở BƯỚC 2 BÊN DƯỚI.')
    print('======================================================================\n')
else:
    print(f'⚠️ Thông tin cài đặt: {ver_res.stderr or ver_res.stdout}')


In [ ]:
# @title 🎙️ Bước 2: Khởi Chạy Giao Diện Web Gradio UI Huấn Luyện { display-mode: "form" }
import os, sys, shutil, subprocess, json, zipfile, glob, re, time, traceback
from pathlib import Path
import torch
import pandas as pd
import gradio as gr

PIPER_SRC_DIR = os.path.abspath('piper/src/python')
if os.path.exists(PIPER_SRC_DIR) and PIPER_SRC_DIR not in sys.path:
    sys.path.insert(0, PIPER_SRC_DIR)
os.environ['PYTHONPATH'] = f"{PIPER_SRC_DIR}:{os.environ.get('PYTHONPATH', '')}"

BASE_DIR = Path('./piper_workspace').resolve()
DATASET_DIR = BASE_DIR / 'dataset'
WAVS_DIR = DATASET_DIR / 'wav'
TRAIN_DIR = BASE_DIR / 'training_dir'
OUTPUT_DIR = BASE_DIR / 'output'
for d in [BASE_DIR, DATASET_DIR, WAVS_DIR, TRAIN_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

TRAIN_PROCESS = None

def unpack_dataset_zip(zip_file_obj, voice_name_hint):
    try:
        if not zip_file_obj:
            return '❌ Chưa có file zip được chọn.', 'Chưa có dữ liệu', []
        zip_path = zip_file_obj.name if hasattr(zip_file_obj, 'name') else str(zip_file_obj)
        if not zipfile.is_zipfile(zip_path):
            return '❌ File tải lên không đúng định dạng ZIP.', 'Lỗi định dạng', []
        
        # Clean old dataset
        if WAVS_DIR.exists(): shutil.rmtree(WAVS_DIR)
        WAVS_DIR.mkdir(parents=True, exist_ok=True)
        meta_file = DATASET_DIR / 'metadata.csv'
        if meta_file.exists(): meta_file.unlink()
        
        with zipfile.ZipFile(zip_path, 'r') as z:
            for item in z.infolist():
                filename = os.path.basename(item.filename)
                if not filename: continue
                if filename == 'metadata.csv':
                    with z.open(item) as src, open(meta_file, 'wb') as dst:
                        dst.write(src.read())
                elif filename.endswith('.wav'):
                    with z.open(item) as src, open(WAVS_DIR / filename, 'wb') as dst:
                        dst.write(src.read())
        
        if not meta_file.exists():
            return '❌ Không tìm thấy file metadata.csv trong file zip.', 'Thiếu metadata.csv', []
        
        wav_count = len(list(WAVS_DIR.glob('*.wav')))
        # Read metadata
        rows = []
        with open(meta_file, 'r', encoding='utf-8') as f:
            for line in f:
                line = line.strip()
                if not line or '|' not in line: continue
                parts = line.split('|', 1)
                rows.append({'ID': parts[0], 'Văn bản (Tiếng Việt)': parts[1]})
        
        v_name = voice_name_hint
        if not v_name or v_name == 'giong_moi':
            base_zip_name = Path(zip_path).stem.replace('_dataset', '')
            v_name = re.sub(r'[^a-zA-Z0-9_-]', '_', base_zip_name).lower()
        
        summary = f'✅ Nạp thành công gói dữ liệu! Tìm thấy {wav_count} file âm thanh WAV (22050Hz) và {len(rows)} câu thoại trong metadata.csv.'
        status = f'Sẵn sàng huấn luyện: {len(rows)} câu thoại ({wav_count} audio clips)'
        return summary, status, rows, v_name
    except Exception as e:
        return f'❌ Lỗi giải nén dataset: {str(e)}', 'Lỗi xử lý', [], voice_name_hint

def start_training(voice_name, epochs, batch_size, resume_from_base, progress=gr.Progress()):
    global TRAIN_PROCESS
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'
        
        meta_file = DATASET_DIR / 'metadata.csv'
        if not meta_file.exists() or not list(WAVS_DIR.glob('*.wav')):
            return '❌ Chưa có dữ liệu! Vui lòng tải file dataset.zip ở trên trước.', 'Chưa có dataset'
        
        progress(0.1, desc='Đang tiền xử lý ngữ âm Tiếng Việt (Preprocess with piper-phonemize)...')
        jsonl_path = DATASET_DIR / 'dataset.jsonl'
        
        # Preprocess command with Vietnamese phonemizer
        prep_cmd = [
            sys.executable, '-m', 'piper_train.preprocess',
            '--language', 'vi',
            '--input-dir', str(DATASET_DIR),
            '--output-dir', str(DATASET_DIR),
            '--dataset-format', 'ljspeech',
            '--single-speaker'
        ]
        prep_res = subprocess.run(prep_cmd, capture_output=True, text=True, env=os.environ)
        if prep_res.returncode != 0:
            return f'❌ Lỗi khi tiền xử lý ngữ âm: {prep_res.stderr or prep_res.stdout}', 'Lỗi tiền xử lý'
        
        progress(0.3, desc='Đang khởi tạo cấu hình Piper Training...')
        train_output_dir = TRAIN_DIR / v_id
        train_output_dir.mkdir(parents=True, exist_ok=True)
        
        # Base checkpoint check
        base_ckpt = Path('base_checkpoint/vi_base.ckpt')
        base_cfg = Path('base_checkpoint/config.json')
        
        # Build train command
        train_cmd = [
            sys.executable, '-m', 'piper_train',
            '--dataset-dir', str(DATASET_DIR),
            '--accelerator', 'gpu' if torch.cuda.is_available() else 'cpu',
            '--devices', '1',
            '--batch-size', str(int(batch_size)),
            '--max_epochs', str(int(epochs)),
            '--checkpoint-epochs', '1',
            '--quality', 'medium'
        ]
        
        if resume_from_base and base_ckpt.exists():
            train_cmd.extend(['--resume_from_checkpoint', str(base_ckpt)])
        
        progress(0.4, desc=f'Đang huấn luyện mạng VITS trên GPU ({epochs} epochs)...')
        print(f"Executing: {' '.join(train_cmd)}")
        
        proc = subprocess.Popen(
            train_cmd,
            cwd=str(train_output_dir),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ
        )
        TRAIN_PROCESS = proc
        
        logs = []
        current_epoch = 0
        for line in proc.stdout:
            clean_line = line.strip()
            if not clean_line: continue
            logs.append(clean_line)
            if len(logs) > 40: logs.pop(0)
            
            # Track epoch
            m = re.search(r'Epoch (\d+)', clean_line)
            if m:
                current_epoch = int(m.group(1))
                pct = min(0.95, 0.4 + 0.55 * (current_epoch / max(1, epochs)))
                progress(pct, desc=f'Đang huấn luyện: Epoch {current_epoch}/{epochs}...')
        
        proc.wait()
        if proc.returncode != 0:
            return f'⚠️ Huấn luyện hoàn thành có cảnh báo: \n' + '\n'.join(logs[-15:]), 'Hoàn thành với cảnh báo'
        
        progress(1.0, desc='✅ Huấn luyện hoàn tất thành công!')
        return f'✅ Đã huấn luyện thành công {epochs} epochs cho giọng "{voice_name}"! Bạn có thể chuyển sang Tab "2. Xuất ONNX" để tải mô hình.', 'Huấn luyện hoàn tất'
    except Exception as e:
        return f'❌ Lỗi huấn luyện: {str(e)}\n{traceback.format_exc()}', 'Lỗi huấn luyện'

def export_onnx_model(voice_name, progress=gr.Progress()):
    try:
        v_id = re.sub(r'[^a-zA-Z0-9_-]', '_', voice_name or 'my_voice').lower().strip('_')
        if not v_id: v_id = 'my_voice'
        
        train_dir = TRAIN_DIR / v_id
        # Find checkpoints
        ckpts = sorted(glob.glob(str(train_dir / '**/*.ckpt'), recursive=True), key=os.path.getmtime)
        if not ckpts:
            # Check base checkpoint fallback
            base_ckpt = Path('base_checkpoint/vi_base.ckpt')
            if base_ckpt.exists():
                ckpts = [str(base_ckpt)]
            else:
                return '❌ Chưa tìm thấy checkpoint nào trong thư mục huấn luyện.', None, None
        
        latest_ckpt = ckpts[-1]
        progress(0.4, desc=f'Đang xuất ONNX từ checkpoint: {os.path.basename(latest_ckpt)}...')
        
        out_onnx = OUTPUT_DIR / f'{v_id}.onnx'
        out_json = OUTPUT_DIR / f'{v_id}.onnx.json'
        
        export_cmd = [
            sys.executable, '-m', 'piper_train.export_onnx',
            latest_ckpt,
            str(out_onnx)
        ]
        exp_res = subprocess.run(export_cmd, capture_output=True, text=True, env=os.environ)
        if exp_res.returncode != 0 or not out_onnx.exists():
            return f'❌ Lỗi xuất ONNX: {exp_res.stderr or exp_res.stdout}', None, None
        
        # Ensure config json exists
        cfg_path = train_dir / 'config.json'
        if not cfg_path.exists():
            cfg_path = Path('base_checkpoint/config.json')
        if cfg_path.exists():
            shutil.copy(str(cfg_path), str(out_json))
        
        progress(1.0, desc='✅ Đã xuất mô hình ONNX thành công!')
        return f'✅ Đã xuất thành công:\n- {out_onnx.name} ({out_onnx.stat().st_size / (1024*1024):.1f} MB)\n- {out_json.name}', str(out_onnx), str(out_json)
    except Exception as e:
        return f'❌ Lỗi khi xuất ONNX: {str(e)}', None, None

# Build Gradio Web Interface
custom_css = '''
.gradio-container { max-width: 960px !important; margin: auto; font-family: system-ui, sans-serif; }
.main-btn { background: linear-gradient(135deg, #f59e0b, #ea580c) !important; color: #020617 !important; font-weight: 700 !important; }
'''

with gr.Blocks(theme=gr.themes.Soft(), css=custom_css, title='Piper Voice Clone Studio') as demo:
    gr.Markdown(
        '''
        # 🎙️ Piper Vietnamese Voice Training Studio
        **Huấn luyện mạng nơ-ron VITS Tiếng Việt trên GPU T4 siêu tốc (Không cần biên dịch Whisper).**
        '''
    )
    
    with gr.Tabs():
        # TAB 1: DATASET & TRAINING
        with gr.TabItem('1. Nạp Dataset & Bắt Đầu Huấn Luyện (Train)'):
            gr.Markdown('### 📦 Bước 1: Nạp file `<ten_giong>_dataset.zip` đã chuẩn bị offline từ TTS Studio')
            with gr.Row():
                dataset_zip_input = gr.File(label='Kéo thả file dataset.zip tại đây', file_types=['.zip'])
            
            voice_name_input = gr.Textbox(label='Tên Giọng / Voice ID', value='giong_moi', placeholder='VD: hoang_nam, lan_anh')
            btn_load_dataset = gr.Button('🔍 Kiểm Tra Dữ Liệu', variant='secondary')
            dataset_status = gr.Markdown('Chờ nạp file dữ liệu...')
            dataset_table = gr.Dataframe(headers=['ID', 'Văn bản (Tiếng Việt)'], label='Xem trước câu thoại', interactive=False)
            
            gr.Markdown('### ⚙️ Bước 2: Cấu hình Huấn Luyện (Fine-tuning)')
            with gr.Row():
                epochs_slider = gr.Slider(minimum=5, maximum=100, value=25, step=5, label='Số Epochs (Khuyên dùng: 20-30)')
                batch_slider = gr.Slider(minimum=4, maximum=32, value=16, step=4, label='Batch Size (GPU T4)')
            
            resume_check = gr.Checkbox(label='Tinh chỉnh từ Checkpoint Tiếng Việt Gốc (Khuyên dùng - converge sau vài phút)', value=True)
            
            btn_start_train = gr.Button('🚀 BẮT ĐẦU HUẤN LUYỆN (TRAIN MODEL)', elem_classes='main-btn', size='lg')
            train_output_status = gr.Textbox(label='Trạng Thái Huấn Luyện', interactive=False)
            
            btn_load_dataset.click(
                fn=unpack_dataset_zip,
                inputs=[dataset_zip_input, voice_name_input],
                outputs=[dataset_status, train_output_status, dataset_table, voice_name_input]
            )
            dataset_zip_input.change(
                fn=unpack_dataset_zip,
                inputs=[dataset_zip_input, voice_name_input],
                outputs=[dataset_status, train_output_status, dataset_table, voice_name_input]
            )
            btn_start_train.click(
                fn=start_training,
                inputs=[voice_name_input, epochs_slider, batch_slider, resume_check],
                outputs=[train_output_status, dataset_status]
            )
        
        # TAB 2: EXPORT ONNX
        with gr.TabItem('2. Xuất File Mô Hình (.onnx & .onnx.json)'):
            gr.Markdown(
                '''
                ### ⚡ Xuất mô hình nén ONNX để nạp vào Piper TTS Studio
                Sau khi quá trình huấn luyện hoàn tất (hoặc đạt số epoch mong muốn), nhấn nút bên dưới để xuất 2 file:
                - `<ten_giong>.onnx`: Trọng số nơ-ron mạng VITS
                - `<ten_giong>.onnx.json`: Cấu hình ngữ âm & mẫu âm học
                '''
            )
            btn_export = gr.Button('⚡ XUẤT FILE .ONNX & .ONNX.JSON', elem_classes='main-btn', size='lg')
            export_status = gr.Markdown('Chờ xuất file...')
            with gr.Row():
                file_onnx_out = gr.File(label='File Mô Hình (.onnx)')
                file_json_out = gr.File(label='File Cấu Hình (.onnx.json)')
            
            btn_export.click(
                fn=export_onnx_model,
                inputs=[voice_name_input],
                outputs=[export_status, file_onnx_out, file_json_out]
            )

demo.queue().launch(share=True, debug=False)
